# SVM: digit classification with linear and RBF kernels

This notebook uses the Optical Digits dataset from the `models.md` table to classify handwritten digits (0–9).
It compares linear and RBF SVMs after feature scaling.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# ---------- Load and scale ----------
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaler on training data only to avoid leakage.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# ---------- Linear SVM ----------
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_sc, y_train)
y_pred_lin = svm_linear.predict(X_test_sc)

print('=== Linear SVM ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lin):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lin, average="weighted"):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_lin, average="weighted"):.4f}')
print(f'F1 score:  {f1_score(y_test, y_pred_lin, average="weighted"):.4f}')

# ---------- RBF SVM ----------
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_sc, y_train)
y_pred_rbf = svm_rbf.predict(X_test_sc)

print('\n=== RBF SVM ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rbf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rbf, average="weighted"):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_rbf, average="weighted"):.4f}')
print(f'F1 score:  {f1_score(y_test, y_pred_rbf, average="weighted"):.4f}')
print('\nClassification report (RBF):\n',
      classification_report(y_test, y_pred_rbf,
                            target_names=[str(d) for d in range(10)]))

# ---------- Confusion matrices side by side ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lin),
                       display_labels=[str(d) for d in range(10)]
                       ).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Linear SVM')

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rbf),
                       display_labels=[str(d) for d in range(10)]
                       ).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('RBF SVM')

plt.suptitle('SVM: confusion matrices', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nLinear vs RBF accuracy: {accuracy_score(y_test, y_pred_lin):.4f} vs '
      f'{accuracy_score(y_test, y_pred_rbf):.4f}')

In [ ]:
# Display a grid of 15 sample predictions for RBF SVM
fig, axes = plt.subplots(3, 5, figsize=(10, 7))
axes = axes.ravel()
for i in range(15):
    img = X_test[i].reshape(8, 8)
    axes[i].imshow(img, cmap='gray')
    true_label = y_test[i]
    pred_label = y_pred_rbf[i]
    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(f'P: {pred_label} | T: {true_label}', color=color)
    axes[i].axis('off')
plt.suptitle('Sample Predictions - RBF SVM', fontsize=16)
plt.tight_layout()
plt.show()


## SVM: margin, kernels, loss, and evaluation

### Notation

- $\mathbf{x}_i \in \mathbb{R}^D$: feature vector ($D = 64$ for Optical Digits).
- $y_i \in \{-1, +1\}$: binary class label (extended to multiclass via one-vs-one).
- $\mathbf{w}$: weight vector; $b$: bias term.
- $\alpha_i$: Lagrange multiplier (dual variable) for sample $i$.
- $C$: regularization parameter controlling the margin–misclassification trade-off.

### Hard-margin SVM

Find the hyperplane that separates the classes with maximum margin:

$$\text{margin} = \frac{2}{\|\mathbf{w}\|}$$

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \qquad \text{s.t.} \quad y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1 \;\; \forall i$$

### Soft-margin SVM and hinge loss

When data is not perfectly separable, introduce slack variables $\xi_i \geq 0$:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{N}\xi_i \qquad \text{s.t.} \quad y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1 - \xi_i$$

Equivalently, using the **hinge loss**:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{N}\max\bigl(0,\; 1 - y_i(\mathbf{w}\cdot\mathbf{x}_i + b)\bigr)$$

**Minimize** the objective: the first term maximizes the margin, the second penalizes misclassifications.

### The kernel trick

Map inputs to a higher-dimensional space $\phi(\mathbf{x})$ where they become linearly separable. The kernel function computes inner products in that space without explicitly computing $\phi$:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)\cdot\phi(\mathbf{x}_j)$$

**Linear kernel:**

$$K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i \cdot \mathbf{x}_j$$

**RBF (Radial Basis Function) kernel:**

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\!\left(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2\right), \qquad \gamma = \frac{1}{2\sigma^2}$$

The RBF kernel maps to an infinite-dimensional space, making it very flexible.

### Dual formulation

The SVM optimization is often solved in its dual form:

$$\max_{\boldsymbol{\alpha}} \sum_{i=1}^{N}\alpha_i - \frac{1}{2}\sum_{i=1}^{N}\sum_{j=1}^{N}\alpha_i\alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j)$$

$$\text{s.t.} \quad 0 \leq \alpha_i \leq C, \qquad \sum_{i=1}^{N}\alpha_i y_i = 0$$

Points with $\alpha_i > 0$ are **support vectors** — they define the decision boundary.

### Multiclass extension

scikit-learn's `SVC` uses the **one-vs-one** strategy: it trains $\binom{K}{2}$ binary classifiers for $K$ classes and uses majority voting for prediction.

### Why feature scaling is critical

SVMs compute distances (via kernels) and margins. If features have different scales, those with larger magnitudes dominate the kernel computation and the margin. StandardScaler ensures all features contribute equally.

### Key hyperparameters

| Parameter | Typical values | Effect |
| --- | --- | --- |
| `C` | 0.01–100 | Regularization: small $C$ = wide margin (more misclassifications); large $C$ = narrow margin (fewer misclassifications) |
| `kernel` | `'linear'`, `'rbf'`, `'poly'` | Kernel function; RBF is most common for non-linear problems |
| `gamma` | `'scale'`, `'auto'`, or float | RBF bandwidth: small $\gamma$ = smooth boundary; large $\gamma$ = complex boundary |
| `degree` | 2–5 | Polynomial kernel degree (only for `kernel='poly'`) |

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

**Maximize** all four metrics: each ranges from $0$ (worst) to $1$ (best). For multiclass problems, weighted averaging across classes accounts for class imbalance.